In [22]:
from singingmaterials.dos_stats import dos_stats_analysis

In [23]:
dos_dict = dos_stats_analysis(mp_id="mp-6947")

Fetching from existing file...


In [27]:
len(dos_dict['projection']['total']['densities'])

1244

In [3]:
dos_dict["element"]["O"]["stats"]

{'athermal': {'band_centre': np.float64(18023660846319.195),
  'integrated_dos': np.float64(11999989153529.512),
  'quantile_25': np.float64(13153427832960.146),
  'quantile_75': np.float64(22625122720474.227),
  'IQR': np.float64(9471694887514.08),
  'shannon_entropy': np.float64(6.619612348742812),
  'densities': array([8.25970807e-05, 7.29928113e-05, 5.75790676e-05, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00], shape=(1244,))}}

In [16]:
from collections import defaultdict
from singingmaterials.dos_stats import phonon_band_centre

element_dos = defaultdict(lambda: 0)
freqs = dos_dict["projection"]["total"]["frequencies"]

for site, data in dos_dict["projection"].items():
    if site == "total":
        continue
    element = site.split("_")[0]          # "O_1" -> "O"
    element_dos[element] = element_dos[element] + data["densities"]

element_band_centres = {el: phonon_band_centre(freqs, dos) for el, dos in element_dos.items()}
print({el: f"{c/1e12:.3f} THz" for el, c in element_band_centres.items()})

{'Si': '15.495 THz', 'O': '18.024 THz'}


In [17]:
band_centres = {
    site: data["stats"]["athermal"]["band_centre"]
    for site, data in dos_dict["projection"].items()
    if site != "total"
}

for site, centre in band_centres.items():
    print(f"{site}: {centre / 1e12:.3f} THz")   # band_centre is in Hz

Si_1: 15.447 THz
Si_2: 15.543 THz
O_1: 18.045 THz
O_2: 18.002 THz
O_3: 18.045 THz
O_4: 18.002 THz


In [4]:
for element, data in dos_dict["element"].items():
    band_centre = data["stats"]["athermal"]["band_centre"]   # in Hz
    print(f"{element}: {band_centre / 1e12:.3f} THz")

Si: 15.495 THz
O: 18.024 THz


In [ ]:

import numpy as np






In [10]:
element_data

{'Si': {'band_centre': np.float64(15495074609967.727),
  'atomic_weight': 28.0855,
  'drum': 'sh'},
 'O': {'band_centre': np.float64(18023660846319.195),
  'atomic_weight': 15.9994,
  'drum': 'sh'}}

In [8]:
drum_map

{'H': 'hh',
 'He': 'hh',
 'Li': 'hh',
 'Be': 'hh',
 'B': 'hh',
 'C': 'hh',
 'N': 'hh',
 'O': 'sh',
 'F': 'sh',
 'Ne': 'sh',
 'Na': 'sh',
 'Mg': 'sh',
 'Al': 'sh',
 'Si': 'sh',
 'P': 'oh',
 'S': 'oh',
 'Cl': 'oh',
 'K': 'oh',
 'Ar': 'oh',
 'Ca': 'oh',
 'Sc': 'oh',
 'Ti': 'rd',
 'V': 'rd',
 'Cr': 'rd',
 'Mn': 'rd',
 'Fe': 'rd',
 'Ni': 'rd',
 'Co': 'rd',
 'Cu': 'cr',
 'Zn': 'cr',
 'Ga': 'cr',
 'Ge': 'cr',
 'As': 'cr',
 'Se': 'cr',
 'Br': 'cr',
 'Kr': 'cb',
 'Rb': 'cb',
 'Sr': 'cb',
 'Y': 'cb',
 'Zr': 'cb',
 'Nb': 'cb',
 'Mo': 'rim',
 'Tc': 'rim',
 'Ru': 'rim',
 'Rh': 'rim',
 'Pd': 'rim',
 'Ag': 'rim',
 'Cd': 'cp',
 'In': 'cp',
 'Sn': 'cp',
 'Sb': 'cp',
 'I': 'cp',
 'Te': 'cp',
 'Xe': 'sd',
 'Cs': 'sd',
 'Ba': 'sd',
 'La': 'sd',
 'Ce': 'sd',
 'Pr': 'sd',
 'Nd': 'ht',
 'Pm': 'ht',
 'Sm': 'ht',
 'Eu': 'ht',
 'Gd': 'ht',
 'Tb': 'ht',
 'Dy': 'mt',
 'Ho': 'mt',
 'Er': 'mt',
 'Tm': 'mt',
 'Yb': 'mt',
 'Lu': 'mt',
 'Hf': 'lt',
 'Ta': 'lt',
 'W': 'lt',
 'Re': 'lt',
 'Os': 'lt',
 'Ir': 'lt',
 'Pt':

In [21]:
import numpy as np
from pymatgen.core import Element

# Strudel's default drum names, ordered roughly from highest to lowest sound.
# The order is a judgement call (cymbals and noise don't have a clear pitch), so swap them around to taste.
DRUMS_HIGH_TO_LOW = ["hh", "oh", "rd", "cr", "cb", "rim", "cp", "sd", "ht", "mt", "lt", "bd"]

def make_drum_map(*element_arrays, lightest="H", heaviest="Bi", drums=DRUMS_HIGH_TO_LOW):
    """Map elements to drums, lightest -> highest drum, heaviest -> lowest drum.

    element_arrays : one or more numpy arrays of element symbols (e.g. one per material).
    lightest : element symbol setting the light end of the mapping (default "H").
        Pass None to use the lightest element in element_arrays instead.
    heaviest : element symbol setting the heavy end of the mapping (default "Bi").
        Pass None to use the heaviest element in element_arrays instead.

    Every element in the periodic table between the two boundary masses is sorted by atomic mass
    and split into len(drums) equal-sized groups. Returns {symbol: drum}.
    """
    symbols = np.unique(np.concatenate([np.asarray(a).ravel() for a in element_arrays]))
    masses = np.array([float(Element(s).atomic_mass) for s in symbols])

    m_min = masses.min() if lightest is None else float(Element(lightest).atomic_mass)
    m_max = masses.max() if heaviest is None else float(Element(heaviest).atomic_mass)
    if masses.max() > m_max:
        raise ValueError(f"{symbols[masses.argmax()]} is heavier than the heaviest boundary ({heaviest})")
    if masses.min() < m_min:
        raise ValueError(f"{symbols[masses.argmin()]} is lighter than the lightest boundary ({lightest})")

    table = [e for e in Element if m_min <= float(e.atomic_mass) <= m_max]
    table.sort(key=lambda e: float(e.atomic_mass))
    if len(table) < len(drums):
        raise ValueError(f"only {len(table)} elements in range, fewer than the {len(drums)} drums")

    return {
        e.symbol: drum
        for drum, group in zip(drums, np.array_split(np.array(table, dtype=object), len(drums)))
        for e in group
    }

def get_strudel_string(dos_dict):
    element_data = {
        el: {
            "band_centre": d["stats"]["athermal"]["band_centre"],   # Hz
            "atomic_weight": float(Element(el).atomic_mass),         # amu
        }
        for el, d in dos_dict["element"].items()
    }

    mats = [np.array(list(dos_stats_analysis(mp_id=m)["element"])) for m in ["mp-6947", "mp-22862", "mp-66", "mp-22862"]]
    drum_map = make_drum_map(*mats)                 # lightest element given → Bi

    for el, info in element_data.items():
        info["drum"] = drum_map[el]

    for el, info in element_data.items():
        info["hits"] = band_centre_to_hits(info["band_centre"])

    layers = ", ".join(f'{info["drum"]}*{info["hits"]}' for info in element_data.values())
    return f'setcpm(30)\ns("{layers}")'


dos_dict = dos_stats_analysis(mp_id="mp-3953")
strudel_string = get_strudel_string(dos_dict)
print(strudel_string)

Fetching from existing file...
Fetching from existing file...
Fetching from existing file...
Fetching from existing file...
Fetching from existing file...
setcpm(30)
s("rd*6, hh*16, oh*11")
